# validation check — team-archetype grid → LR (polynomial), HOME market

A different **region definition** from the 4D-matchup notebook, on the shared
`evaluation.model_check` framework. Instead of clustering the whole match on four numbers at once:

1. cluster individual **teams** in 2D (their own goals for / against) into a few archetypes,
2. label each match by the **pair** (home archetype, away archetype) — a grid of type-vs-type cells,
3. keep the cells where the home win beat the market on the training fold,
4. fit the model and bet the home win where it beats the implied probability by `buffer`.

The grid names every type-vs-type combination explicitly, so it can isolate a pocket (e.g. two
low-scoring teams) that the flat 4D clustering might merge with its neighbours. `running` vs `final`
again refers only to which stats the **team archetypes** are fit on; matches are always assigned, and
the model fit, on running stats.

In [ ]:
# --- SETUP -------------------------------------------------------------------
# Load the wide ABT for one league group and pull in the model-check framework. Everything that
# never changes between checks (folds, ROI scoring, the honest per-fold engine, the three gates,
# the registry write) lives in `evaluation.model_check`; this notebook only defines the parts that
# DO change — the space, the target, the segmenter, and the model.
GROUP = "major"   # league group: which <group>/ ABT to validate
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import make_pipeline

from evaluation.model_check import (
    run_check, evaluate, collect_bets, pooled_roi, roi_grouped,
    season_folds, expanding_folds,
)

abt = pd.read_parquet(f"C:/Users/twluc/frame/01_data/03_abt/{GROUP}/abt.parquet")

# Attach each team's end-of-season goals for/against to every match it played, once as the home
# side and once as the away side. These columns feed only the "final" cluster-boundary variant; a
# match is never LOCATED by them (that would leak the season's outcome into its own features).
final_stats = pd.read_parquet(f"C:/Users/twluc/frame/01_data/02_features/{GROUP}/team_season_final.parquet")
final_stats = final_stats[["team", "season", "season_goals_for_avg", "season_goals_agst_avg"]]
home_final = final_stats.rename(columns={"team": "hmt_name",
    "season_goals_for_avg": "home_for_final", "season_goals_agst_avg": "home_agst_final"})
away_final = final_stats.rename(columns={"team": "awt_name",
    "season_goals_for_avg": "away_for_final", "season_goals_agst_avg": "away_agst_final"})
abt = abt.merge(home_final, on=["hmt_name", "season"], how="left").merge(away_final, on=["awt_name", "season"], how="left")

In [ ]:
# --- SPACE (modifiable) ------------------------------------------------------
# The bet-signal model still sees the full 4D matchup (both teams' running for/against). The team
# CLUSTERING, however, looks at one team at a time in 2D — hence the per-side 2D maps below.
RUNNING_FEATURES = [
    "hmt_season_goals_for_avg", "hmt_season_goals_agst_avg",
    "awt_season_goals_for_avg", "awt_season_goals_agst_avg",
]
# One team in 2D (goals for, goals against), running (point-in-time) and final (end-of-season) forms.
HOME_RUN, AWAY_RUN = ["hmt_season_goals_for_avg", "hmt_season_goals_agst_avg"], ["awt_season_goals_for_avg", "awt_season_goals_agst_avg"]
HOME_FIN, AWAY_FIN = ["home_for_final", "home_agst_final"], ["away_for_final", "away_agst_final"]

In [ ]:
# --- TARGET (modifiable) -----------------------------------------------------
# The one cell that differs between the draw and home versions of this notebook.
OUTCOME = "t_home_flg"           # 1 when the home win happened
IMPLIED = "b365_home_impl"       # implied prob we bet against = 1 / b365 odds
MRKT_IMPLIED = "mrkt_home_impl"  # market-consensus implied prob, for profitability only

In [ ]:
# --- ABT FILTERING -----------------------------------------------------------
MIN_GAME_NUMBER = 8
matches = abt[
    (abt["hmt_game_number"] > MIN_GAME_NUMBER)
    & (abt["awt_game_number"] > MIN_GAME_NUMBER)
].dropna(subset=RUNNING_FEATURES + HOME_FIN + AWAY_FIN + [IMPLIED, MRKT_IMPLIED, OUTCOME, "league", "season"]).copy()

print(f"matches: {len(matches)}   seasons: {sorted(matches['season'].unique())}   "
      f"leagues: {sorted(matches['league'].unique())}")

In [ ]:
# --- SEGMENTER (modifiable) --------------------------------------------------
# Same contract as always: segmenter(train) -> labeler ; labeler(rows) -> Series, NaN = don't bet.
# This one clusters individual TEAMS into archetypes, then labels each match by its (home, away) cell
# and keeps the cells that beat the market on train. `home_map`/`away_map` pick which stats define the
# archetypes; cells are always assigned by the RUNNING per-side stats, so bet-time labelling is honest.

# Keep the segments (clusters / cells / categories) where the target happened MORE often than the
# market priced it — computed on the TRAIN fold only, so the held-out fold never influences the
# choice. This is the "profitable-segment" selection rule; a rule-based segmenter would skip it.
def keep_profitable(train, labels, min_matches=100):
    frame = pd.DataFrame({"label": labels.to_numpy(),
                          "won": train[OUTCOME].to_numpy(),
                          "implied": train[MRKT_IMPLIED].to_numpy()})
    stats = frame.groupby("label").agg(won=("won", "mean"), implied=("implied", "mean"), n=("won", "size"))
    return set(stats.index[(stats["n"] >= min_matches) & (stats["won"] > stats["implied"])])

def team_grid_segmenter(home_map, away_map, n_types, min_matches=100):
    def segmenter(train):
        # archetypes from both sides stacked, training seasons only
        team_points = np.vstack([train[home_map].to_numpy(), train[away_map].to_numpy()])
        km = KMeans(n_clusters=n_types, random_state=0, n_init=10).fit(team_points)
        def cells(rows):   # a match's cell = "home_type-away_type", always assigned by running stats
            h = km.predict(rows[HOME_RUN].to_numpy())
            a = km.predict(rows[AWAY_RUN].to_numpy())
            return pd.Series([f"{i}-{j}" for i, j in zip(h, a)], index=rows.index)
        kept = keep_profitable(train, cells(train), min_matches)
        def labeler(rows):
            c = cells(rows)
            return c.where(c.isin(kept))     # non-kept cells -> NaN -> not bet
        return labeler
    return segmenter

N_TEAM_TYPES = 4   # archetypes -> N_TEAM_TYPES**2 grid cells (reference; swept below as a diagnostic)
SEGMENTERS = {
    "running": team_grid_segmenter(HOME_RUN, AWAY_RUN, N_TEAM_TYPES),
    "final":   team_grid_segmenter(HOME_FIN, AWAY_FIN, N_TEAM_TYPES),
}

In [ ]:
# --- MODEL (modifiable) ------------------------------------------------------
# The bet-signal model: given the space features, predict the probability of the target outcome.
# Today it is polynomial-expanded features (squares + pairwise products), standardised, then logistic
# regression. To try a different method (random forest, XGBoost, ...) swap this one factory — the
# only contract the framework needs is an sklearn-style estimator with .fit(X, y) and .predict_proba(X).
def make_model():
    return make_pipeline(PolynomialFeatures(degree=2, include_bias=False),
                         StandardScaler(), LogisticRegression(max_iter=3000))

BUFFER = 0.02   # only bet when the model probability beats the implied one by at least this margin

## Gates 1–3, running vs final

In [ ]:
for name, seg in SEGMENTERS.items():
    r = evaluate(matches, features=RUNNING_FEATURES, outcome=OUTCOME, implied=IMPLIED,
                 segmenter=seg, make_model=make_model, buffer=BUFFER)
    if r["empty"]:
        print(f"[{name:7s}] no bets placed"); continue
    g = lambda ok: "PASS" if ok else "FAIL"
    print(f"[{name:7s}] pooled {r['pooled']:+.4f}  ({len(r['bets'])} bets)   "
          f"G1 {g(r['g1'])} (+{int((r['per_season'] > 0).sum())}/{len(r['per_season'])})   "
          f"G2 {g(r['g2'])} (drop {r['best_league']} -> {r['without_league'][r['best_league']]:+.3f})   "
          f"G3 {g(r['g3'])}")
    print("          seasons:  " + "  ".join(f"{s}:{v:+.3f}" for s, v in r["per_season"].items()))
    print("          walk-fwd: " + ("  ".join(f"{s}:{v:+.3f}" for s, v in r["wf"].items())
                                     if not r["walk"].empty else "no bets"))

## Knob robustness — sweep the number of team archetypes
Not a gate. Cells = types². The edge should survive a range of archetype counts.

In [ ]:
print("pooled ROI (b365) by number of team archetypes (cells = types**2):")
print("  types  cells  running   final")
for n in [2, 3, 4, 5]:
    common = dict(features=RUNNING_FEATURES, outcome=OUTCOME, implied=IMPLIED,
                  make_model=make_model, buffer=BUFFER)
    run = collect_bets(matches, season_folds(matches),
                       segmenter=team_grid_segmenter(HOME_RUN, AWAY_RUN, n), **common)
    fin = collect_bets(matches, season_folds(matches),
                       segmenter=team_grid_segmenter(HOME_FIN, AWAY_FIN, n), **common)
    print(f"    {n}     {n**2:2d}   {pooled_roi(run['implied'], run['won']):+.4f}  "
          f"{pooled_roi(fin['implied'], fin['won']):+.4f}")

## Descriptive (in-sample, not a gate) — ROI per grid cell
Which type-vs-type cell drives the home win market? Archetypes fit on all matches (running),
each cell annotated with the two archetypes' (goals for, goals against). In-sample — a map of where
to look, not evidence.

In [ ]:
team_points = np.vstack([matches[HOME_RUN].to_numpy(), matches[AWAY_RUN].to_numpy()])
km = KMeans(n_clusters=N_TEAM_TYPES, random_state=0, n_init=10).fit(team_points)
print("team archetypes (for, agst):")
for i, centre in enumerate(km.cluster_centers_):
    print(f"  type {i}: for={centre[0]:.2f}  agst={centre[1]:.2f}")

home_type = km.predict(matches[HOME_RUN].to_numpy())
away_type = km.predict(matches[AWAY_RUN].to_numpy())
grid = pd.DataFrame({"cell": [f"{h}-{a}" for h, a in zip(home_type, away_type)],
                     "won": matches[OUTCOME].astype(int).to_numpy(),
                     "implied": matches[IMPLIED].to_numpy()})
per_cell = grid.groupby("cell").agg(won=("won", "sum"), staked=("implied", "sum"), n=("won", "size"))
per_cell["roi"] = (per_cell["won"] - per_cell["staked"]) / per_cell["staked"]
print("\nROI per cell (home-type - away-type), in-sample b365:")
print(per_cell[["n", "roi"]].sort_values("roi", ascending=False).round(3).to_string())

## Diagnostic — where the edge lives (league × season), running map

In [ ]:
r = evaluate(matches, features=RUNNING_FEATURES, outcome=OUTCOME, implied=IMPLIED,
             segmenter=SEGMENTERS["running"], make_model=make_model, buffer=BUFFER)
breakdown = roi_grouped(r["bets"], ["league", "season"]).unstack("season").round(3)
print(f"ROI by league x season (b365), running map, {len(r['bets'])} bets:")
print(breakdown.to_string())

## Summary

In [ ]:
run_check(matches, features=RUNNING_FEATURES, outcome=OUTCOME, implied=IMPLIED,
          segmenter=SEGMENTERS["running"], make_model=make_model, buffer=BUFFER,
          source="goals_foragst_home_kmeans_team_lr", space="goals_foragst", target="home",
          abt_filter="major, gn>8, dropna",
          segmentation=f"KMeans k={N_TEAM_TYPES} team archetypes -> {N_TEAM_TYPES}x{N_TEAM_TYPES} grid",
          segment_filter="train-profitable cells")